PIFOCast, a simple weather prediction model
===========================================


##### Copyright 2025 Nicolas Gasnier.

In [1]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

##### Introduction

This project is based on the following :
- Google Deepmind [GraphCast](https://www.science.org/doi/10.1126/science.adi2336) and [GenCast](https://arxiv.org/abs/2312.15796) papers, and associated [source code](https://github.com/google-deepmind/graphcast) as a source of inspiration
- Tensorflow GNN's colab example ["Learning shortest path with GraphNetowks in TF-GNN"](https://colab.research.google.com/github/tensorflow/gnn/blob/master/examples/notebooks/graph_network_shortest_path.ipynb#scrollTo=qr1_8ttC08vu), from wich the base Encoder / Processor / Decoder architecture is derived.


In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import math
import collections
import functools
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_gnn as tfgnn
from tensorflow_gnn import runner
from typing import Callable, Optional, Mapping

import importlib

import cdsapi

# Theses two rows for improving development workflow of our module
import pifocast
importlib.reload(pifocast)

from pifocast import LatLonGrid, pifo, buildGridGNN, getGraphExample, getGraphForFeatures, get_dataset, load_dataset, pifoGridGenerator


2025-06-09 09:58:00.946009: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-09 09:58:00.949569: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-09 09:58:00.960613: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749455880.979660   26844 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749455880.985174   26844 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749455881.000256   26844 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

### Data retrieval from Era5

In [3]:
# era5dataset = "reanalysis-era5-pressure-levels"

# years = ["2024"]
# monthes = ["01", "02", "03",
#         "04", "05", "06",
#         "07", "08", "09",
#         "10", "11", "12"]
# days = [
#     "01", "02", "03",
#     "04", "05", "06",
#     "07", "08", "09",
#     "10", "11", "12",
#     "13", "14", "15",
#     "16", "17", "18",
#     "19", "20", "21",
#     "22", "23", "24",
#     "25", "26", "27",
#     "28", "29", "30",
#     "31"
# ]
# times = [
#     "00:00", "06:00",
#     #"12:00", "18:00"
# ]

# for year in years:
#     for month in monthes:
#         request = {
#             "product_type": ["reanalysis"],
#             "variable": [
#                 "geopotential",
#                 "u_component_of_wind",
#                 "v_component_of_wind"
#             ],
#             "year": [year],
#             "month": [month],                
#             "day": days,
#             "time": times,
#             "pressure_level": ["500"],
#             "data_format": "grib",
#             "download_format": "unarchived"
#         }

#         target = 'dataset/'+year+month+'.grib'

#         client = cdsapi.Client()
#         client.retrieve(era5dataset, request, target)

### Dataset generation - building the whole dataset from Era5

In [4]:
schema = tfgnn.read_schema("pifo.pbtxt")
graph_spec = tfgnn.create_graph_spec_from_schema_pb(schema)
grid = LatLonGrid("dataset/202504.grib", 4)



2025-06-09 09:58:08.680371: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
trainFiles = [
    "202401.grib", "202402.grib", "202403.grib",
    "202404.grib", "202405.grib", "202406.grib",
    "202407.grib", "202408.grib", "202409.grib",
    "202410.grib", "202411.grib", "202412.grib"
]


In [8]:
def writeExamples(gribFile, x_train_DSWriter, y_train_DSWriter, x_test_DSWriter, y_test_DSWriter):
    i = 1
    nb_examples = 0
    nb_validations = 0
    for graph, target in pifoGridGenerator(gribFile, grid, step=2):
        #graph, target = getGraphExample(grid)
        example = tfgnn.write_example(graph)
        if i>25:
            x_test_DSWriter.write(example.SerializeToString())
            nb_validations += 1
        else:
            nb_examples += 1
            x_train_DSWriter.write(example.SerializeToString())
        targetSR = tf.io.serialize_tensor(target)
        targetFeature = {
            'target' : tf.train.Feature(bytes_list=tf.train.BytesList(value=[targetSR.numpy()]))
        }    
        targetExample = tf.train.Example(features=tf.train.Features(feature=targetFeature))
        if i>25:
            y_test_DSWriter.write(targetExample.SerializeToString())
        else:
            y_train_DSWriter.write(targetExample.SerializeToString())
        i += 1

    return (nb_examples, nb_validations)


In [9]:

with tf.io.TFRecordWriter("dataset/x_train_ds.tfrecord") as x_train_DSWriter, tf.io.TFRecordWriter("dataset/y_train_ds.tfrecord") as y_train_DSWriter:
    with tf.io.TFRecordWriter("dataset/x_test_ds.tfrecord") as x_test_DSWriter, tf.io.TFRecordWriter("dataset/y_test_ds.tfrecord") as y_test_DSWriter:
        total_nb_examples = 0
        total_nb_validations = 0
        for grib in trainFiles:
            print("Processing ", grib)
            (nb_examples, nb_validations) = writeExamples("dataset/"+grib, x_train_DSWriter, y_train_DSWriter, x_test_DSWriter, y_test_DSWriter)
            total_nb_examples += nb_examples
            total_nb_validations += nb_validations
        print("nb examples", total_nb_examples)
        print("nb validations", total_nb_validations)


Processing  202401.grib
    grib index  0
    grib index  2
    grib index  4
    grib index  6
    grib index  8
    grib index  10
    grib index  12
    grib index  14
    grib index  16
    grib index  18
    grib index  20
    grib index  22
    grib index  24
    grib index  26
    grib index  28
    grib index  30
    grib index  32
    grib index  34
    grib index  36
    grib index  38
    grib index  40
    grib index  42
    grib index  44
    grib index  46
    grib index  48
    grib index  50
    grib index  52
    grib index  54
    grib index  56
    grib index  58
    grib index  60
Processing  202402.grib
    grib index  0
    grib index  2
    grib index  4
    grib index  6
    grib index  8
    grib index  10
    grib index  12
    grib index  14
    grib index  16
    grib index  18
    grib index  20
    grib index  22
    grib index  24
    grib index  26
    grib index  28
    grib index  30
    grib index  32
    grib index  34
    grib index  36
    grib ind

In [10]:
trainDS = load_dataset(graph_spec, "dataset/x_train_ds.tfrecord", "dataset/y_train_ds.tfrecord")
print("listing train DS...")
#print(next(iter(graphDS)))
for i, graph in enumerate(trainDS.batch(2).take(3)):
     print(f"Input {i}: {graph}")

listing train DS...
Input 0: (GraphTensor(
  context=Context(features={}, sizes=[[1]
 [1]], shape=(2,), indices_dtype=tf.int32),
  node_set_names=['grid'],
  edge_set_names=['edges']), <tf.Tensor: shape=(2, 65160, 3), dtype=float32, numpy=
array([[[0.31320038, 0.41739193, 0.5123773 ],
        [0.31320038, 0.41739193, 0.5123773 ],
        [0.31320038, 0.41739193, 0.5123773 ],
        ...,
        [0.2938077 , 0.41739193, 0.5123773 ],
        [0.2938077 , 0.41739193, 0.5123773 ],
        [0.2938077 , 0.41739193, 0.5123773 ]],

       [[0.22802769, 0.4173876 , 0.5123736 ],
        [0.22802769, 0.4173876 , 0.5123736 ],
        [0.22802769, 0.4173876 , 0.5123736 ],
        ...,
        [0.2680021 , 0.4173876 , 0.5123736 ],
        [0.2680021 , 0.4173876 , 0.5123736 ],
        [0.2680021 , 0.4173876 , 0.5123736 ]]], dtype=float32)>)
Input 1: (GraphTensor(
  context=Context(features={}, sizes=[[1]
 [1]], shape=(2,), indices_dtype=tf.int32),
  node_set_names=['grid'],
  edge_set_names=['edges'

2025-06-09 10:39:30.370593: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:381] TFRecordDataset `buffer_size` is unspecified, default to 262144
2025-06-09 10:39:30.563313: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
testDS = load_dataset(graph_spec, "dataset/x_test_ds.tfrecord", "dataset/y_test_ds.tfrecord")
print("listing test DS...")
#print(next(iter(graphDS)))
for i, graph in enumerate(testDS.batch(2).take(3)):
     print(f"Input {i}: {graph}")

listing test DS...
Input 0: (GraphTensor(
  context=Context(features={}, sizes=[[1]
 [1]], shape=(2,), indices_dtype=tf.int32),
  node_set_names=['grid'],
  edge_set_names=['edges']), <tf.Tensor: shape=(2, 65160, 3), dtype=float32, numpy=
array([[[0.170355  , 0.4173894 , 0.51236486],
        [0.170355  , 0.4173894 , 0.51236486],
        [0.170355  , 0.4173894 , 0.51236486],
        ...,
        [0.2444819 , 0.4173894 , 0.51236486],
        [0.2444819 , 0.4173894 , 0.51236486],
        [0.2444819 , 0.4173894 , 0.51236486]],

       [[0.40753773, 0.41739616, 0.51237184],
        [0.40753773, 0.41739616, 0.51237184],
        [0.40753773, 0.41739616, 0.51237184],
        ...,
        [0.28082913, 0.41739616, 0.51237184],
        [0.28082913, 0.41739616, 0.51237184],
        [0.28082913, 0.41739616, 0.51237184]]], dtype=float32)>)
Input 1: (GraphTensor(
  context=Context(features={}, sizes=[[1]
 [1]], shape=(2,), indices_dtype=tf.int32),
  node_set_names=['grid'],
  edge_set_names=['edges']

2025-06-09 10:39:37.306633: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
